# 03 – Modelling

Project: Developer Salary Prediction  
Data: `data/processed/survey_clean_2025.csv` (7,428 employed professional developers in Europe)

Goal: predict a developer's annual salary from their profile, and find the model with the lowest error.

Plan for this notebook:
- prepare the features based on the EDA in notebook 02,
- split the data and keep the test set aside until the final evaluation,
- compare two simple baselines and four models on a validation set,
- choose the model with the lowest MAE (tuning and the final test follow later).

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Seed used everywhere, for reproducibility
SEED = 1

sns.set_theme(style="whitegrid")

## Load the data

In [ ]:
df = pd.read_csv("../data/processed/survey_clean_2025.csv")

target = "ConvertedCompYearly"
orgsize_order = ["<20", "20-99", "100-499", "500-999", "1,000-4,999", "5,000-9,999", "10,000+"]
age_order = ["18-24", "25-34", "35-44", "45-54", "55+"]

print(df.shape)

## Features

Choices from the EDA:
- `WorkExp` is used as log(WorkExp + 1): salary rises fast in the first years and then levels off (B1, T5). Tree models aren't affected by this transformation, so one version works for all models.
- `YearsCode` is dropped (correlation 0.88 with `WorkExp`, F1). Instead I add `coding_before_work` = `YearsCode` − `WorkExp`, the years of coding before working life (negative for career changers).
- `Age` stays: at the same experience it still relates to salary (F2).

In [ ]:
# New features
# ------------
df["WorkExp_log"] = np.log1p(df["WorkExp"])
df["coding_before_work"] = df["YearsCode"] - df["WorkExp"]

# Feature groups
# --------------
numeric_cols = ["WorkExp_log", "coding_before_work", "n_languages"]
onehot_cols = ["Country", "DevType", "Industry", "RemoteWork", "ICorPM", "EdLevel"]
ordinal_cols = ["OrgSize", "Age"]
lang_cols = [col for col in df.columns if col.startswith("lang_")]

feature_cols = numeric_cols + onehot_cols + ordinal_cols + ["languages_missing"] + lang_cols

X = df[feature_cols]
y = df[target]

print("Features before encoding:", X.shape[1])
df[["coding_before_work"]].describe().round(1).T

## Train-test split

20% of the data is kept as a test set and stays untouched until the final evaluation. Everything that learns from the data (language selection, fill values, encoder, scaler) is fitted on the training set only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

## Language selection

Only languages used by at least 5% of the training set (people with language data) are kept. Rare languages don't give the model enough examples to learn from.

In [ ]:
lang_share = X_train.loc[X_train["languages_missing"] == 0, lang_cols].mean()
selected_langs = lang_share[lang_share >= 0.05].index.tolist()

print(f"Languages kept: {len(selected_langs)} of {len(lang_cols)}")
print([col.replace("lang_", "") for col in selected_langs])

## Missing values

- Numeric columns: median of the training set.
- One-hot columns: missing becomes its own category, "unknown".
- Ordered columns (`OrgSize`, `Age`): "unknown" has no place in the order, so the most frequent value of the training set is used.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

# Numeric: training median
numeric_medians = X_train[numeric_cols].median()
X_train[numeric_cols] = X_train[numeric_cols].fillna(numeric_medians)
X_test[numeric_cols] = X_test[numeric_cols].fillna(numeric_medians)

# One-hot columns: "unknown" category
X_train[onehot_cols] = X_train[onehot_cols].fillna("unknown")
X_test[onehot_cols] = X_test[onehot_cols].fillna("unknown")

# Ordered columns: most frequent value in training
ordinal_modes = X_train[ordinal_cols].mode().iloc[0]
X_train[ordinal_cols] = X_train[ordinal_cols].fillna(ordinal_modes)
X_test[ordinal_cols] = X_test[ordinal_cols].fillna(ordinal_modes)

print("Missing values left (train / test):", X_train.isnull().sum().sum(), "/", X_test.isnull().sum().sum())
print("\nFill values:")
print(pd.concat([numeric_medians, ordinal_modes]).to_string())

## Encoding

`OrgSize` and `Age` are encoded as ordered numbers, because salary follows their order (B7, B10). `EdLevel` is one-hot encoded together with the other categorical columns, because "Associate" breaks the order (B6).

In [ ]:
# Ordinal encoding (fixed order)
# ------------------------------
orgsize_map = {category: i for i, category in enumerate(orgsize_order)}
age_map = {category: i for i, category in enumerate(age_order)}

X_train_ordinal = pd.DataFrame({
    "OrgSize_encoded": X_train["OrgSize"].map(orgsize_map),
    "Age_encoded": X_train["Age"].map(age_map)})
X_test_ordinal = pd.DataFrame({
    "OrgSize_encoded": X_test["OrgSize"].map(orgsize_map),
    "Age_encoded": X_test["Age"].map(age_map)})

# One-hot encoding
# ----------------
encoder = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")

# fit + transform on train, transform only on test
X_train_onehot = encoder.fit_transform(X_train[onehot_cols])
X_test_onehot = encoder.transform(X_test[onehot_cols])

onehot_names = encoder.get_feature_names_out(onehot_cols)
X_train_onehot_df = pd.DataFrame(X_train_onehot, columns=onehot_names, index=X_train.index)
X_test_onehot_df = pd.DataFrame(X_test_onehot, columns=onehot_names, index=X_test.index)

print("One-hot columns:", len(onehot_names))

## Scaling

The numeric and ordinal features are standardized (needed for the linear model; tree models don't depend on it). The 0/1 columns stay as they are.

In [ ]:
to_scale_train = pd.concat([X_train[numeric_cols], X_train_ordinal], axis=1)
to_scale_test = pd.concat([X_test[numeric_cols], X_test_ordinal], axis=1)

scaler = StandardScaler()

# fit + transform on train, transform only on test
X_train_scaled = pd.DataFrame(scaler.fit_transform(to_scale_train),
                              columns=to_scale_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(to_scale_test),
                             columns=to_scale_test.columns, index=X_test.index)

## Final feature table

In [ ]:
X_train_final = pd.concat([X_train_scaled, X_train_onehot_df,
                           X_train[["languages_missing"] + selected_langs]], axis=1)
X_test_final = pd.concat([X_test_scaled, X_test_onehot_df,
                          X_test[["languages_missing"] + selected_langs]], axis=1)

print(f"X_train_final shape: {X_train_final.shape}")
print(f"X_test_final shape:  {X_test_final.shape}")
print("Missing values:", X_train_final.isnull().sum().sum(), "/", X_test_final.isnull().sum().sum())

## Validation split

The test set stays untouched until the final evaluation, so I compare the models on a validation set: 20% of the training data.

Limitation: the fill values, encoder and scaler were fitted on the whole training set, so they have seen the validation rows. The effect is small, and the test set is completely separate.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final, y_train,
    test_size=0.2,
    random_state=SEED)

print(f"X_tr shape:  {X_tr.shape}")
print(f"X_val shape: {X_val.shape}")

## Evaluation

For every baseline and model I record:
- R² on training and validation (a big gap means overfitting),
- MAE: the average error in USD (main metric),
- RMSE: like MAE, but large errors count more.

Models trained on log salary have their predictions converted back to USD with `np.exp()` before scoring, so all results are comparable.

In [ ]:
results = {}

def evaluate(name, y_pred_tr, y_pred_val):
    """Print the scores of one model and store them in results."""
    scores = {
        "R2 (train)": r2_score(y_tr, y_pred_tr),
        "R2 (validation)": r2_score(y_val, y_pred_val),
        "MAE": mean_absolute_error(y_val, y_pred_val),
        "RMSE": mean_squared_error(y_val, y_pred_val) ** 0.5,
    }
    results[name] = scores
    print(f"{name}")
    print(f"  R2 (train):      {scores['R2 (train)'] * 100:.2f}%")
    print(f"  R2 (validation): {scores['R2 (validation)'] * 100:.2f}%")
    print(f"  MAE:  {scores['MAE']:,.0f} USD")
    print(f"  RMSE: {scores['RMSE']:,.0f} USD")

## Baselines

Two simple rules that any model has to beat:
- everyone gets the median salary of the training data,
- everyone gets the median salary of their country. Country is the strongest factor in the EDA, so this rule already captures a lot.

In [ ]:
# General median
# --------------
median_salary = y_tr.median()

evaluate("Baseline: general median",
         np.full(len(y_tr), median_salary),
         np.full(len(y_val), median_salary))

In [ ]:
# Country median
# --------------
country_tr = X_train.loc[X_tr.index, "Country"]
country_val = X_train.loc[X_val.index, "Country"]

country_medians = y_tr.groupby(country_tr).median()

evaluate("Baseline: country median",
         country_tr.map(country_medians),
         country_val.map(country_medians).fillna(median_salary))

## Models

Four models, from simple to complex, each trained on the raw and on the log salary, with default settings (tuning comes later):
- `SGDRegressor`: linear model,
- `DecisionTreeRegressor`: a single tree,
- `RandomForestRegressor`: many trees, averaged,
- `XGBRegressor`: trees built one after another, each fixing the errors of the previous ones.

What I expect from the EDA:
- the country baseline is already much better than the general median,
- the linear model struggles more than the trees, because of the curve in experience and the age × experience interaction,
- a single tree overfits (training R² far above validation R²), and the forest and XGBoost do better.

In [ ]:
# SGDRegressor
# ------------
# With the default tol, SGD stopped too early on the log target, before it had learned
# the average log salary. A smaller tol and more iterations let it converge.
sgd_raw = SGDRegressor(loss="squared_error", max_iter=5000, tol=1e-5, random_state=SEED)
sgd_raw.fit(X_tr, y_tr)
evaluate("SGDRegressor (raw)", sgd_raw.predict(X_tr), sgd_raw.predict(X_val))

sgd_log = SGDRegressor(loss="squared_error", max_iter=5000, tol=1e-5, random_state=SEED)
sgd_log.fit(X_tr, np.log(y_tr))
evaluate("SGDRegressor (log)", np.exp(sgd_log.predict(X_tr)), np.exp(sgd_log.predict(X_val)))

In [ ]:
# Decision Tree
# -------------
tree_raw = DecisionTreeRegressor(random_state=SEED)
tree_raw.fit(X_tr, y_tr)
evaluate("Decision Tree (raw)", tree_raw.predict(X_tr), tree_raw.predict(X_val))

tree_log = DecisionTreeRegressor(random_state=SEED)
tree_log.fit(X_tr, np.log(y_tr))
evaluate("Decision Tree (log)", np.exp(tree_log.predict(X_tr)), np.exp(tree_log.predict(X_val)))

In [ ]:
# Random Forest
# -------------
rf_raw = RandomForestRegressor(random_state=SEED)
rf_raw.fit(X_tr, y_tr)
evaluate("Random Forest (raw)", rf_raw.predict(X_tr), rf_raw.predict(X_val))

rf_log = RandomForestRegressor(random_state=SEED)
rf_log.fit(X_tr, np.log(y_tr))
evaluate("Random Forest (log)", np.exp(rf_log.predict(X_tr)), np.exp(rf_log.predict(X_val)))

In [ ]:
# XGBoost
# -------
xgb_raw = XGBRegressor(random_state=SEED)
xgb_raw.fit(X_tr, y_tr)
evaluate("XGBoost (raw)", xgb_raw.predict(X_tr), xgb_raw.predict(X_val))

xgb_log = XGBRegressor(random_state=SEED)
xgb_log.fit(X_tr, np.log(y_tr))
evaluate("XGBoost (log)", np.exp(xgb_log.predict(X_tr)), np.exp(xgb_log.predict(X_val)))

## Comparison

In [ ]:
results_df = pd.DataFrame(results).T.sort_values("MAE")
results_df.style.format({"R2 (train)": "{:.3f}", "R2 (validation)": "{:.3f}", "MAE": "{:,.0f}", "RMSE": "{:,.0f}"})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=results_df["MAE"], y=results_df.index, ax=ax)
ax.set_xlabel("MAE on the validation set (USD)")
ax.set_ylabel("")
ax.set_title("Average error by model")
plt.tight_layout()
plt.show()

Notes: *(to be evaluated together)*

## Next steps
- Tune the chosen model, with cross-validation (`cross_val_score`), RandomizedSearchCV and GridSearchCV
- Final evaluation on the test set
- Error analysis and feature importance